In [3]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss
from sklearn.base import is_classifier
from sklearn.ensemble import RandomForestRegressor # Added for robust tabular surrogate
from dataclasses import dataclass
from typing import List, Iterator

DATA_NPZ_PATH = "../../datasets/yield_preprocessed.npz"
OUT_DIR = "./outputs_local"
os.makedirs(OUT_DIR, exist_ok=True)

N_TEST_SAMPLES = 50
SAMPLE_SEED = 123
RANDOM_STATE = 42

repo_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import existing implementations
from cgt_xai.models import get_model
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.banzhaf import exact_banzhaf_index

def load_strict_data(npz_path):
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"NPZ not found at: {npz_path}")

    print(f"Loading data from {npz_path}...")
    d = np.load(npz_path, allow_pickle=True)

    try:
        X_train = d["X_train"].astype(float)
        X_test  = d["X_test"].astype(float)
        y_train = d["y_train"].astype(int).ravel()
        y_test  = d["y_test"].astype(int).ravel()
    except KeyError as e:
        raise KeyError(f"Missing expected key in NPZ: {e}. Keys found: {list(d.keys())}")

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    print(f"Shapes -> X_train: {X_train.shape}, y_train: {y_train.shape}, X_test: {X_test.shape}")

    assert X_train.ndim == 2, "X_train must be 2D"
    assert X_test.ndim == 2,  "X_test must be 2D"
    assert X_train.shape[0] == y_train.shape[0], f"Mismatch: X_train rows {X_train.shape[0]} != y_train {y_train.shape[0]}"
    assert X_test.shape[0] == y_test.shape[0], f"Mismatch: X_test rows {X_test.shape[0]} != y_test {y_test.shape[0]}"
    assert len(feature_names) == X_train.shape[1], "Mismatch between feature_names length and X columns"

    return X_train, y_train, X_test, y_test, feature_names

X_train, y_train, X_test, y_test, feature_names = load_strict_data(DATA_NPZ_PATH)
n_features = X_train.shape[1]

all_labels = sorted(np.unique(y_train).tolist())
print(f"Labels detected: {all_labels}")

trained_models = {}
MODEL_NAMES = ["logistic", "svm", "random_forest"]

print("\nTraining models...")
for m_name in MODEL_NAMES:
    model = get_model(m_name, random_state=RANDOM_STATE)
    if not is_classifier(model):
         print(f"Warning: {m_name} does not seem to be a classifier (is_classifier=False).")
    model.fit(X_train, y_train)
    trained_models[m_name] = model

print("Models trained:", list(trained_models.keys()))

print("\nTraining Surrogate Models...")
X_means = np.mean(X_train, axis=0)
AUGMENT_FACTOR = 10
X_aug = np.tile(X_train, (AUGMENT_FACTOR, 1))
n_aug_samples = X_aug.shape[0]

# Generate random masks using the paper's specified sampling distribution
mask_rng = np.random.default_rng(RANDOM_STATE)
masks = np.zeros_like(X_aug)
for i in range(n_aug_samples):
    k = mask_rng.integers(0, n_features + 1)
    if k > 0:
        idx = mask_rng.choice(n_features, size=k, replace=False)
        masks[i, idx] = 1

# Mask the features: Keep original if mask=1, replace with mean if mask=0
X_masked = X_aug * masks + X_means * (1 - masks)

# Concatenate the masked features and the mask vector itself
X_surrogate_train = np.hstack([X_masked, masks])

surrogate_models = {}
for m_name, original_model in trained_models.items():
    print(f"  Training surrogate for {m_name}...")

    # Target is the original model's soft predictions
    if hasattr(original_model, "predict_proba"):
        Y_target = original_model.predict_proba(X_aug)
    else:
        Y_target = original_model.predict(X_aug).reshape(-1, 1)

    # Using RandomForest instead of MLP for better tabular data support
    surrogate = RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    surrogate.fit(X_surrogate_train, Y_target)
    surrogate_models[m_name] = surrogate

print("Surrogate models trained successfully.")

@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self) -> int: return len(self.indices)
    def __iter__(self) -> Iterator[int]: return iter(self.indices)
    def __getitem__(self, i: int) -> int: return self.indices[i]

class SurrogateExpectationUtility:
    def __init__(self, surrogate_model, x_full, y_true, X_means, feature_subset, feature_names, all_labels):
        self.surrogate = surrogate_model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.X_means = np.asarray(X_means, dtype=float).ravel()
        self.n_features = len(X_means)

        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.class_idx = self.all_labels.index(self.y_true)

        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[feature_names[j] for j in feature_subset]
        )

    def __call__(self, S):
        # 1. Create the binary mask for the current coalition
        mask = np.zeros(self.n_features)
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            mask[global_j] = 1

        # 2. Mask the input instance (use means for excluded features)
        x_masked = self.x_full * mask + self.X_means * (1 - mask)

        # 3. Concatenate and reshape for the surrogate prediction
        x_in = np.hstack([x_masked, mask]).reshape(1, -1)

        # 4. Predict the expected probabilities
        expected_probs = self.surrogate.predict(x_in)

        # Match the original Log-Loss Utility for a fair comparison
        if expected_probs.ndim == 2:
            # Clip to avoid log(0) errors and normalize
            probs = np.clip(expected_probs, 1e-15, 1 - 1e-15)
            probs = probs / np.sum(probs, axis=1, keepdims=True)
            return -float(log_loss([self.y_true], probs, labels=self.all_labels))
        else:
            # Fallback for hard predictions
            return 0.0 if (expected_probs[0] > 0.5) == self.y_true else -1.0

rng = np.random.default_rng(SAMPLE_SEED)
test_idx = rng.choice(len(X_test), size=min(N_TEST_SAMPLES, len(X_test)), replace=False)

METHODS = [
    ("shapley", exact_shapley_value),
    ("least_core", exact_least_core),
    ("nucleolus", exact_nucleolus),
    ("gately", exact_gately_value),
    ("banzhaf", exact_banzhaf_index),
]

records = []

print(f"\nRunning explanation on {len(test_idx)} samples using {len(METHODS)} methods...")

for model_name, model in trained_models.items():
    if model_name == "svm" and not getattr(model, "probability", True) and not hasattr(model, "predict_proba"):
        print(f"Warning: SVM model might lack predict_proba. Hard label fallbacks apply.")

    for i, idx in enumerate(test_idx):
        x = X_test[idx]
        y = y_test[idx]

        feat_subset = np.arange(n_features)

        # Use the newly trained Surrogate Utility
        util = SurrogateExpectationUtility(
            surrogate_model=surrogate_models[model_name],
            x_full=x,
            y_true=y,
            X_means=X_means,
            feature_subset=feat_subset,
            feature_names=feature_names,
            all_labels=all_labels
        )

        for method_name, method_fn in METHODS:
            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                alloc = np.asarray(alloc, dtype=float).ravel()

                alloc_named = {feature_names[j]: float(alloc[j]) for j in range(n_features)}

                rec = {
                    "model": model_name,
                    "test_index": int(idx),
                    "y_true": int(y),
                    "method": method_name,
                    "alloc_full": alloc.tolist(),
                    "alloc_named": alloc_named
                }
                records.append(rec)
            except Exception as e:
                print(f"Error in {method_name} (Model: {model_name}, Idx: {idx}): {e}")

df = pd.DataFrame(records)
print("\nExperiment Complete. First 5 rows:")
print(df.head())

out_file = os.path.join(OUT_DIR, "surrogate_expectation_results.csv")
df.to_csv(out_file, index=False)
print(f"\nResults saved to: {out_file}")

Loading data from ../datasets/yield_preprocessed.npz...
Shapes -> X_train: (800, 4), y_train: (800,), X_test: (200, 4)
Labels detected: [0, 1, 2]

Training models...
Models trained: ['logistic', 'svm', 'random_forest']

Training Surrogate Models...
  Training surrogate for logistic...
  Training surrogate for svm...
  Training surrogate for random_forest...
Surrogate models trained successfully.

Running explanation on 50 samples using 5 methods...

Experiment Complete. First 5 rows:
      model  test_index  y_true      method  \
0  logistic          34       0     shapley   
1  logistic          34       0  least_core   
2  logistic          34       0   nucleolus   
3  logistic          34       0      gately   
4  logistic          34       0     banzhaf   

                                          alloc_full  \
0  [0.06512803696875254, 0.2199295330607785, 0.20...   
1  [-0.19321847878373158, -0.12866947362668807, -...   
2  [-0.21073506928693614, -0.14680190887962177, -...   
3  [

In [ ]:
import os
import sys
import warnings
import logging
import json
from pathlib import Path
from typing import Dict, List, Callable, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import log_loss # Added for utility
import quantus

# Setup Paths
sys.path.append(os.path.abspath(os.path.join("..", "..")))

# Import GT methods
from cgt_xai.concepts.shapley import exact_shapley_value
from cgt_xai.concepts.least_core import exact_least_core
from cgt_xai.concepts.nucleolus import exact_nucleolus
from cgt_xai.concepts.gately_point import exact_gately_value
from cgt_xai.concepts.shapley_shubik import exact_shapley_shubik
from cgt_xai.concepts.banzhaf import exact_banzhaf_index
from cgt_xai.models import get_model

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Config
DATA_NPZ_PATH = "../../datasets/yield_preprocessed.npz"
RESULTS_CSV_PATH = "./outputs_local/surrogate_expectation_results.csv"
OUT_DIR = Path("./quantus_conditional/quantus_evaluation_soil")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
AOPC_TOPK_LIST = [1, 2, 3]


@dataclass
class FeaturePlayers:
    indices: List[int]
    data_names: List[str]
    def __len__(self): return len(self.indices)
    def __iter__(self): return iter(self.indices)
    def __getitem__(self, i): return self.indices[i]

class SurrogateExpectationUtility:
    def __init__(self, surrogate_model, x_full, y_true, X_means, feature_subset, feature_names, all_labels):
        self.surrogate = surrogate_model
        self.x_full = np.asarray(x_full, dtype=float).ravel()
        self.X_means = np.asarray(X_means, dtype=float).ravel()
        self.n_features = len(X_means)

        self.y_true = int(y_true)
        self.all_labels = all_labels
        self.class_idx = self.all_labels.index(self.y_true)

        self.feature_subset = np.asarray(feature_subset, dtype=int).ravel()
        self.data = FeaturePlayers(
            indices=list(range(len(feature_subset))),
            data_names=[str(feature_names[j]) for j in feature_subset]
        )

    def __call__(self, S):
        mask = np.zeros(self.n_features)
        for local_j in S:
            global_j = self.feature_subset[int(local_j)]
            mask[global_j] = 1

        x_masked = self.x_full * mask + self.X_means * (1 - mask)
        x_in = np.hstack([x_masked, mask]).reshape(1, -1)

        expected_probs = self.surrogate.predict(x_in)

        # Calculate Log-Loss to match the scale of the previous baseline
        if expected_probs.ndim == 2:
            probs = np.clip(expected_probs, 1e-15, 1 - 1e-15)
            probs = probs / np.sum(probs, axis=1, keepdims=True)
            return -float(log_loss([self.y_true], probs, labels=self.all_labels))
        else:
            return 0.0 if (expected_probs[0] > 0.5) == self.y_true else -1.0

class SklearnTorchWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.is_clf = hasattr(model, "predict_proba")

    def forward(self, x):
        if isinstance(x, torch.Tensor):
            x_np = x.detach().cpu().numpy()
        else:
            x_np = x

        if x_np.ndim == 3:
            x_np = x_np.reshape(x_np.shape[0], -1)
        elif x_np.ndim == 1:
            x_np = x_np.reshape(1, -1)

        if self.is_clf:
            out = self.model.predict_proba(x_np)
        else:
            out = self.model.predict(x_np)
            if out.ndim == 1: out = out.reshape(-1, 1)

        return torch.from_numpy(out).float()

def load_data_strict():
    d = np.load(DATA_NPZ_PATH, allow_pickle=True)
    X_train = d["X_train"].astype(float)
    X_test = d["X_test"].astype(float)
    y_train = d["y_train"].astype(int).ravel()
    y_test = d["y_test"].astype(int).ravel()

    if "feature_names" in d:
        feature_names = d["feature_names"].tolist()
    else:
        feature_names = [f"f{i}" for i in range(X_train.shape[1])]

    return X_train, X_test, y_train, y_test, feature_names

# 2. robust perturbation & distance functions

def perturb_tabular_robust(arr, indices, baseline, **kwargs):
    arr_pert = arr.copy()
    base_vec = baseline.flatten()

    if arr.ndim == 2:
        F = arr.shape[1]
    elif arr.ndim == 3:
        F = arr.shape[2]
    else:
        F = len(base_vec)

    def apply_mask(target_slice, idxs):
        valid_idxs = [i for i in idxs if i < F]
        if valid_idxs:
            target_slice[valid_idxs] = base_vec[valid_idxs]

    if arr.ndim == 2:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b], idx_list)

    elif arr.ndim == 3:
        B = arr.shape[0]
        for b in range(B):
            idx_list = indices[b] if isinstance(indices[b], (list, np.ndarray)) else [indices[b]]
            apply_mask(arr_pert[b, 0], idx_list)

    return arr_pert

def perturb_uniform_robust(arr, **kwargs):
    lower = kwargs.get('lower_bound', 0.01)
    upper = kwargs.get('upper_bound', 0.02)
    noise = np.random.uniform(lower, upper, size=arr.shape)
    sign = np.random.choice([-1, 1], size=arr.shape)
    return arr + (noise * sign)

def distance_euclidean(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))

# 3. metric definitions

def get_metrics(baseline_vec):
    base_3d = baseline_vec.reshape(1, 1, -1)
    perturb_kwargs = {"baseline": base_3d}

    metrics = {
        "faithfulness_corr": quantus.FaithfulnessCorrelation(
            nr_runs=10,
            subset_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "faithfulness_estimate": quantus.FaithfulnessEstimate(
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "selectivity": quantus.Selectivity(
            patch_size=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "monotonicity": quantus.Monotonicity(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "pixel_flipping": quantus.PixelFlipping(
            features_in_step=1,
            perturb_baseline="mean",
            perturb_func=perturb_tabular_robust,
            perturb_func_kwargs=perturb_kwargs,
            return_aggregate=True
        ),
        "sparseness": quantus.Sparseness(return_aggregate=True),
        "complexity": quantus.Complexity(return_aggregate=True),
        "local_lipschitz_estimate": quantus.LocalLipschitzEstimate(
            nr_samples=5, perturb_std=0.1,
            norm_numerator=distance_euclidean,
            norm_denominator=distance_euclidean,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        ),
        "max_sensitivity": quantus.MaxSensitivity(
            nr_samples=5, lower_bound=0.1, upper_bound=0.2,
            norm_numerator=quantus.norm_func.fro_norm,
            norm_denominator=quantus.norm_func.fro_norm,
            perturb_func=perturb_uniform_robust,
            perturb_func_kwargs={},
            return_aggregate=True
        )
    }
    return metrics

def compute_aopc_manual(sk_model, X_test_2d, y_test, attributions_2d, top_k_list, baseline):
    if hasattr(sk_model, "predict_proba"):
        base_probs = sk_model.predict_proba(X_test_2d)
        base_scores = base_probs[np.arange(len(y_test)), y_test]
    else:
        return np.nan

    drops_all_k = []
    for k in top_k_list:
        if k > X_test_2d.shape[1]: continue
        X_perturbed = X_test_2d.copy()
        for i in range(len(X_test_2d)):
            top_indices = np.argsort(-np.abs(attributions_2d[i]))[:k]
            X_perturbed[i, top_indices] = baseline[top_indices]
        new_probs = sk_model.predict_proba(X_perturbed)
        new_scores = new_probs[np.arange(len(y_test)), y_test]
        drops_all_k.append(np.mean(base_scores - new_scores))
    return np.mean(drops_all_k)


def make_explain_func(method_name, surrogate_model, X_means, all_labels, feature_names):
    METHOD_MAP = {
        "shapley": exact_shapley_value,
        "least_core": exact_least_core,
        "nucleolus": exact_nucleolus,
        "gately": exact_gately_value,
        "shapley_shubik": exact_shapley_shubik,
        "banzhaf": exact_banzhaf_index
    }
    method_fn = METHOD_MAP.get(method_name)

    def explain_func(model, inputs, targets, **kwargs):
        # We ignore 'model' because we use the pre-trained 'surrogate_model'
        attributions = []
        for i in range(len(inputs)):
            x_in = inputs[i]
            if hasattr(x_in, "detach"):
                x_in = x_in.detach().cpu().numpy()
            x = x_in.flatten()

            y_in = targets[i]
            y = y_in.item() if hasattr(y_in, 'item') else y_in

            util = SurrogateExpectationUtility(
                surrogate_model=surrogate_model,
                x_full=x,
                y_true=y,
                X_means=X_means,
                feature_subset=np.arange(len(x)),
                feature_names=feature_names,
                all_labels=all_labels
            )

            try:
                res = method_fn(util)
                alloc = getattr(res, 'values', res)
                attributions.append(np.asarray(alloc, dtype=float).reshape(1, -1))
            except Exception as e:
                attributions.append(np.zeros((1, len(x))))

        return np.array(attributions)

    return explain_func

# 5. main

def main():
    # 1. Load Data
    X_train, X_test, y_train, y_test, feature_names = load_data_strict()
    baseline = np.mean(X_train, axis=0)
    all_labels = sorted(np.unique(y_train).tolist())
    n_features = X_train.shape[1]

    # 2. Load pre-computed attributions
    df_results = pd.read_csv(RESULTS_CSV_PATH)
    valid_indices = sorted(df_results["test_index"].unique())
    X_subset = X_test[valid_indices]
    y_subset = y_test[valid_indices]

    # Quantus Input: (N, 1, F)
    x_batch = X_subset[:, np.newaxis, :]

    models_to_run = df_results["model"].unique()
    methods_to_run = df_results["method"].unique()

    logger.info(f"Loaded {len(valid_indices)} samples for evaluation.")

    # 3. Pre-train required original and surrogate models
    logger.info("Re-training base and surrogate models for dynamic robustness evaluation...")
    base_models = {}
    surrogate_models = {}

    AUGMENT_FACTOR = 10
    X_aug = np.tile(X_train, (AUGMENT_FACTOR, 1))
    mask_rng = np.random.default_rng(RANDOM_STATE)
    masks = np.zeros_like(X_aug)
    for i in range(X_aug.shape[0]):
        k = mask_rng.integers(0, n_features + 1)
        if k > 0:
            idx = mask_rng.choice(n_features, size=k, replace=False)
            masks[i, idx] = 1
    X_masked = X_aug * masks + baseline * (1 - masks)
    X_surrogate_train = np.hstack([X_masked, masks])

    for model_name in models_to_run:
        sk_model = get_model(model_name, random_state=RANDOM_STATE)
        sk_model.fit(X_train, y_train)
        base_models[model_name] = sk_model

        if hasattr(sk_model, "predict_proba"):
            Y_target = sk_model.predict_proba(X_aug)
        else:
            Y_target = sk_model.predict(X_aug).reshape(-1, 1)

        surrogate = RandomForestRegressor(
            n_estimators=100,
            min_samples_leaf=5,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
        surrogate.fit(X_surrogate_train, Y_target)
        surrogate_models[model_name] = surrogate

    # 4. Evaluation Loop
    all_eval_rows = []
    metrics = get_metrics(baseline)

    for model_name in models_to_run:
        logger.info(f"--- Evaluating Model: {model_name} ---")

        sk_model = base_models[model_name]
        surrogate_model = surrogate_models[model_name]
        torch_model = SklearnTorchWrapper(sk_model).eval()

        df_model = df_results[df_results["model"] == model_name]

        for method_name in methods_to_run:
            logger.info(f"   Method: {method_name}")

            attributions_list = []
            for idx in valid_indices:
                row = df_model[(df_model["test_index"] == idx) & (df_model["method"] == method_name)]
                if row.empty:
                    attributions_list.append(np.zeros(n_features))
                else:
                    val_str = row.iloc[0]["alloc_full"]
                    if isinstance(val_str, str):
                        val_arr = np.array(json.loads(val_str))
                    else:
                        val_arr = np.array(val_str)
                    attributions_list.append(val_arr)

            a_batch_2d = np.array(attributions_list)
            a_batch_3d = a_batch_2d[:, np.newaxis, :]

            row_stats = {"model": model_name, "method": method_name}
            y_faith = y_subset if hasattr(sk_model, "predict_proba") else np.zeros_like(y_subset)

            # Global Metrics
            for m_name, metric_fn in metrics.items():
                if m_name in ["local_lipschitz_estimate", "max_sensitivity"]: continue
                try:
                    scores = metric_fn(
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_faith,
                        a_batch=a_batch_3d,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            # Aopc
            try:
                aopc = compute_aopc_manual(sk_model, X_subset, y_subset, a_batch_2d, AOPC_TOPK_LIST, baseline)
                row_stats["aopc_manual"] = aopc
            except Exception as e:
                 row_stats["aopc_manual"] = np.nan

            # Robustness Metrics (Requires the explainer function and the surrogate model)
            explain_func = make_explain_func(method_name, surrogate_model, baseline, all_labels, feature_names)

            for m_name in ["local_lipschitz_estimate", "max_sensitivity"]:
                try:
                    scores = metrics[m_name](
                        model=torch_model,
                        x_batch=x_batch,
                        y_batch=y_subset,
                        explain_func=explain_func,
                        channel_first=True,
                        softmax=False
                    )
                    row_stats[m_name] = np.nanmean(scores)
                except Exception as e:
                    logger.warning(f"     {m_name} Error: {e}")
                    row_stats[m_name] = np.nan

            all_eval_rows.append(row_stats)
            f_corr = row_stats.get('faithfulness_corr', np.nan)
            ms = row_stats.get('max_sensitivity', np.nan)
            logger.info(f"     -> Faith_Corr: {f_corr:.4f}, MaxSens: {ms:.4f}")

    # 5. Save output
    df_eval = pd.DataFrame(all_eval_rows)
    out_path = OUT_DIR / "quantus_results_full_extended.csv"
    df_eval.to_csv(out_path, index=False)
    print(df_eval.to_string())

if __name__ == "__main__":
    main()

            model      method  faithfulness_corr  faithfulness_estimate  selectivity  monotonicity  pixel_flipping  sparseness  complexity  aopc_manual  local_lipschitz_estimate  max_sensitivity
0        logistic     shapley           0.230964               0.223478     0.425386          0.42        0.425386    0.359449    1.101314     0.276571                  0.013895         0.003547
1        logistic  least_core           0.128809               0.223287     0.425386          0.50        0.425386    0.276742    1.215804     0.309042                  0.032752         0.002489
2        logistic   nucleolus           0.140949               0.223205     0.425386          0.50        0.425386    0.277015    1.214843     0.309042                  0.018211         0.002477
3        logistic      gately           0.207707               0.222234     0.425386          0.50        0.425386    0.274695    1.218909     0.309042                  0.016935         0.002300
4        logistic     ban